# AutoML Experiment Analysis

Analysis of the autonomous AutoML loop recorded in `results.tsv`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_PATH = Path('results.tsv')
if not RESULTS_PATH.exists():
    raise FileNotFoundError('results.tsv not found. Run python prepare.py and log at least one experiment first.')

df = pd.read_csv(RESULTS_PATH, sep='\t')
df['private_score'] = pd.to_numeric(df['private_score'], errors='coerce')
df['public_score'] = pd.to_numeric(df['public_score'], errors='coerce')
df['train_seconds'] = pd.to_numeric(df['train_seconds'], errors='coerce')
df['status'] = df['status'].fillna('').str.strip().str.upper()
df['medal'] = df['medal'].fillna('unavailable').str.strip().str.lower()

print(f'Total experiments: {len(df)}')
df.head()


In [ ]:
counts = df['status'].value_counts()
print('Experiment outcomes:')
print(counts.to_string())

n_keep = counts.get('KEEP', 0)
n_discard = counts.get('DISCARD', 0)
n_crash = counts.get('CRASH', 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f'\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}')
print(f'Crash count: {n_crash}')

medal_counts = df['medal'].value_counts()
print('\nMedal buckets:')
print(medal_counts.to_string())


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
if kept.empty:
    print('No kept experiments yet.')
else:
    best_row = kept.loc[kept['private_score'].idxmax()]
    baseline = kept.iloc[0]
    print(f'Baseline private_score: {baseline.private_score:.6f}')
    print(f'Best private_score:     {best_row.private_score:.6f}')
    print(f'Best public_score:      {best_row.public_score:.6f}')
    print(f'Best medal:             {best_row.medal}')
    print(f'Best description:       {best_row.description}')
    print(f'Improvement vs baseline: {best_row.private_score - baseline.private_score:+.6f}')


## Score Progress

Track the private score, the public proxy score, and the running best frontier.


In [ ]:
plot_df = df.copy().reset_index(drop=True)
plot_df['trial'] = np.arange(1, len(plot_df) + 1)
plot_df['running_best_private'] = plot_df['private_score'].cummax()

fig, ax = plt.subplots(figsize=(16, 8))

status_colors = {'KEEP': '#0f766e', 'DISCARD': '#9f1239', 'CRASH': '#6b7280'}
for status, color in status_colors.items():
    subset = plot_df[plot_df['status'] == status]
    if not subset.empty:
        ax.scatter(subset['trial'], subset['private_score'], label=f'{status.title()} private', color=color, s=50, alpha=0.8)

ax.plot(plot_df['trial'], plot_df['running_best_private'], color='#111827', linewidth=2.5, label='Running best private')
ax.plot(plot_df['trial'], plot_df['public_score'], color='#2563eb', linewidth=1.5, alpha=0.7, label='Public proxy')

ax.set_title('AutoML Progress Over Experiments')
ax.set_xlabel('Experiment Number')
ax.set_ylabel('ROC AUC')
ax.grid(alpha=0.25)
ax.legend()

fig.tight_layout()
fig.savefig('progress.png', dpi=200, bbox_inches='tight')
plt.show()

print('Saved plot to progress.png')


In [ ]:
if kept.empty:
    print('No kept experiments to rank yet.')
else:
    ranked = kept.sort_values('private_score', ascending=False)[['commit', 'private_score', 'public_score', 'train_seconds', 'medal', 'description']]
    ranked.head(15)


In [ ]:
if kept.empty or len(kept) == 1:
    print('Need at least two kept experiments to show stepwise improvements.')
else:
    frontier = kept.copy().reset_index(drop=True)
    frontier['prev_private'] = frontier['private_score'].shift(1)
    frontier['delta_private'] = frontier['private_score'] - frontier['prev_private']
    frontier.iloc[1:][['commit', 'private_score', 'delta_private', 'description']].sort_values('delta_private', ascending=False).head(10)
